In [1]:
!pip install pandas transformers torch datasets protobuf sentencepiece evaluate scikit-learn evaluate accelerate>=0.26.0 tensorboard dotenv


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, pipeline, AutoConfig
import torch
from torch import nn
from datasets import Dataset, DatasetDict
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import evaluate
from datetime import datetime
from huggingface_hub import login
from dotenv import load_dotenv
import os

d:\giorg\corsi\ESGI\5eme_annee\NLP\projet_nlp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

login(HF_TOKEN)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
df = pd.read_csv('../data/raw/bitext-retail-ecommerce-llm-chatbot-training-dataset.csv')
df.head()

,instruction,intent,category,tags,response
0,I got to add an item to the cart,add_product,CART,BL,I'll get right on it! I'm here to assist you i...
1,wanna add fucking products to the basket can h...,add_product,CART,BCIMQWZ,I sincerely apologize if you've encountered an...
2,i have to add products to the basket i ned help,add_product,CART,BCMQZ,You bet! I'm here to assist you in adding prod...
3,di like to add products to the cart could i ge...,add_product,CART,BCILMPQZ,Indeed! I'm here to assist you in adding produ...
4,"I need to add an item to the cart , where do I...",add_product,CART,BCILZ,I'll take care of it! I'm here to help you wit...


In [5]:
df['category'].unique()

array(['CART', 'PRODUCT', 'ORDER', 'ACCOUNT', 'CONTACT', 'DELIVERY',
       'PAYMENT', 'RETURNS', 'USER', 'SALES', 'STORE', 'FEEDBACK',
       'APP_WEBSITE'], dtype=object)

In [6]:
df[df['category'] == 'PRODUCT'].head(1)['instruction']

957    i got to see the availability of an item i nee...
Name: instruction, dtype: object

In [7]:
# je selectionne uniquement les trois premieres categories
small_df = df[df['category'].isin(df['category'].unique()[:3])]
small_df['category'].unique()

array(['CART', 'PRODUCT', 'ORDER'], dtype=object)

In [40]:
# test avec le dataset complet
small_df = df
small_df['category'].unique()

array(['CART', 'PRODUCT', 'ORDER', 'ACCOUNT', 'CONTACT', 'DELIVERY',
       'PAYMENT', 'RETURNS', 'USER', 'SALES', 'STORE', 'FEEDBACK',
       'APP_WEBSITE'], dtype=object)

In [41]:
small_df.head()

,instruction,intent,category,tags,response
0,I got to add an item to the cart,add_product,CART,BL,I'll get right on it! I'm here to assist you i...
1,wanna add fucking products to the basket can h...,add_product,CART,BCIMQWZ,I sincerely apologize if you've encountered an...
2,i have to add products to the basket i ned help,add_product,CART,BCMQZ,You bet! I'm here to assist you in adding prod...
3,di like to add products to the cart could i ge...,add_product,CART,BCILMPQZ,Indeed! I'm here to assist you in adding produ...
4,"I need to add an item to the cart , where do I...",add_product,CART,BCILZ,I'll take care of it! I'm here to help you wit...


In [42]:
# je mantien que les colonnes instruction et category
small_df = small_df[['instruction', 'category']].rename(columns={'instruction': 'text', 'category': 'label'})
small_df.head()

,text,label
0,I got to add an item to the cart,CART
1,wanna add fucking products to the basket can h...,CART
2,i have to add products to the basket i ned help,CART
3,di like to add products to the cart could i ge...,CART
4,"I need to add an item to the cart , where do I...",CART


In [43]:
labels = small_df['label'].unique().tolist()
labels

['CART',
 'PRODUCT',
 'ORDER',
 'ACCOUNT',
 'CONTACT',
 'DELIVERY',
 'PAYMENT',
 'RETURNS',
 'USER',
 'SALES',
 'STORE',
 'FEEDBACK',
 'APP_WEBSITE']

In [44]:
id2label = {i: l for i, l in enumerate(labels)}
print(id2label)

label2id = {l: i for i, l in enumerate(labels)}
print(label2id)

{0: 'CART', 1: 'PRODUCT', 2: 'ORDER', 3: 'ACCOUNT', 4: 'CONTACT', 5: 'DELIVERY', 6: 'PAYMENT', 7: 'RETURNS', 8: 'USER', 9: 'SALES', 10: 'STORE', 11: 'FEEDBACK', 12: 'APP_WEBSITE'}
{'CART': 0, 'PRODUCT': 1, 'ORDER': 2, 'ACCOUNT': 3, 'CONTACT': 4, 'DELIVERY': 5, 'PAYMENT': 6, 'RETURNS': 7, 'USER': 8, 'SALES': 9, 'STORE': 10, 'FEEDBACK': 11, 'APP_WEBSITE': 12}


In [45]:
# je verifie que ma version de CUDA est bien detectee pour utiliser le GPU
torch.cuda.is_available()

True

In [46]:
MODEL_NAME = 'cross-encoder/ms-marco-TinyBERT-L2-v2' # 'meta-llama/Llama-3.2-1B" #"meta-llama/Meta-Llama-3-8B-Instruct"

# Prepare DataFrame: 'text' and 'label' columns
dataset = Dataset.from_pandas(small_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
num_labels = small_df['label'].nunique()

In [ ]:
# 1. Assume you already have a DataFrame `df` with columns: "text" (str) and 'label' (str or int)
# Example:
# df = pd.read_csv("your_data.csv")  # must contain 'text' and 'label'

# Encode string labels to integers if needed
# if small_df['label'].dtype == "O":
#     le = LabelEncoder()
#     print('Original labels:', small_df['label'].head())
#     small_df['label'] = le.fit_transform(small_df['label'])
#     print('Encoded labels:', small_df['label'].head())
# else:
#     le = None  # labels already numeric

print('Original labels:', small_df['label'].head())
small_df["label"] = small_df["label"].map(label2id)
print('Encoded labels:', small_df['label'].head())

num_labels = small_df['label'].nunique()
print(f"Number of labels: {num_labels}")

Original labels: 0    CART
1    CART
2    CART
3    CART
4    CART
Name: label, dtype: object
Encoded labels: 0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64
Number of labels: 13


In [48]:
random_seed = 42
# # 2. Train/validation split
# train_df, val_df = train_test_split(small_df, test_size=0.2, random_state=random_seed, stratify=small_df['label'])

# # 3. Convert pandas -> HF Datasets
# train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
# val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
# datasets = DatasetDict({"train": train_dataset, "validation": val_dataset})


# First split: train vs temp (test+validation)
train_df, t_and_v_df = train_test_split(small_df, random_state=random_seed, test_size=0.2)
# Second split: temp -> test vs validation (10% + 10%)
test_df, val_df = train_test_split(t_and_v_df, random_state=random_seed, test_size=0.5)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

datasets = DatasetDict(
    {
        "train": train_dataset,      # 80%
        "test": test_dataset,       # 10%
        "val": val_dataset,  # 10% (held-out final validation)
    }
)


In [49]:
# 4. Load TinyBERT tokenizer and model (you can swap to any TinyBERT checkpoint)
model_name = "prajjwal1/bert-tiny"  # a common TinyBERT-like model on HF Hub
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized_datasets = datasets.map(tokenize_function, batched=True)

# Set format for PyTorch
tokenized_datasets = tokenized_datasets.remove_columns(
    [c for c in tokenized_datasets["train"].column_names if c not in ["input_ids", "attention_mask", 'label']]
)
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", 'label'])

Map: 100%|██████████| 4489/4489 [00:00<00:00, 11594.92 examples/s]


In [50]:
# 5. Metrics: accuracy and loss will be tracked; we compute accuracy explicitly
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"accuracy": acc}


In [51]:
# 6. Load TinyBERT with a new classification head sized to num_labels

config = AutoConfig.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    config=config,
)

# 6a. Print all layers / parameter names before freezing
# print("Model parameters before freezing:")
# for name, param in model.named_parameters():
#     print(name) #print(name, param.shape)


# 6b. Freeze all layers except the classification head
# for name, param in model.named_parameters():
#     # keep only the classifier (and optionally dropout/biases) trainable
#     if "classifier" in name:
#         param.requires_grad = True
#     else:
#         param.requires_grad = False

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [55]:
finetuned_model_name = "new_tinybert_cls"
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
run_name = f"{finetuned_model_name}_e5_lr1e-5_{run_id}"

# 7. Training configuration
training_args = TrainingArguments(
    output_dir=f"../models/{finetuned_model_name}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,

    # >>> TensorBoard-related args <<<
    logging_dir=f"../logs/{run_name}",   # where TensorBoard will read logs
    report_to=["tensorboard"],           # force logging to TensorBoard
    run_name=run_name
)

# 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 9. Train
trainer.train()

C:\Users\giorg\AppData\Local\Temp\ipykernel_30096\1642837396.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.041800,0.025339,0.994430
2,0.039000,0.019758,0.995321
3,0.044400,0.017607,0.995321
4,0.025800,0.014418,0.995766
5,0.023300,0.013263,0.995989
6,0.025700,0.014315,0.995989
7,0.017700,0.013074,0.995989
8,0.021900,0.012620,0.996212
9,0.015900,0.012463,0.995989
10,0.001900,0.012367,0.996212


TrainOutput(global_step=22450, training_loss=0.022699424735620984, metrics={'train_runtime': 474.1076, 'train_samples_per_second': 757.36, 'train_steps_per_second': 47.352, 'total_flos': 114439756976640.0, 'train_loss': 0.022699424735620984, 'epoch': 10.0})

In [56]:
# final evaluation on the held-out validation set
final_metrics = trainer.evaluate(
    eval_dataset=tokenized_datasets["val"]
)
print("Final validation metrics:", final_metrics)

Final validation metrics: {'eval_loss': 0.016194507479667664, 'eval_accuracy': 0.9964357317888171, 'eval_runtime': 2.6517, 'eval_samples_per_second': 1692.888, 'eval_steps_per_second': 105.97, 'epoch': 10.0}


In [57]:
# 10. Final evaluation: accuracy and validation loss
eval_results = trainer.evaluate()
print("Validation results:", eval_results)

Validation results: {'eval_loss': 0.012620269320905209, 'eval_accuracy': 0.9962121212121212, 'eval_runtime': 3.2779, 'eval_samples_per_second': 1369.175, 'eval_steps_per_second': 85.726, 'epoch': 10.0}


In [58]:
df = pd.read_csv('../data/raw/bitext-retail-ecommerce-llm-chatbot-training-dataset.csv')
df.head()


,instruction,intent,category,tags,response
0,I got to add an item to the cart,add_product,CART,BL,I'll get right on it! I'm here to assist you i...
1,wanna add fucking products to the basket can h...,add_product,CART,BCIMQWZ,I sincerely apologize if you've encountered an...
2,i have to add products to the basket i ned help,add_product,CART,BCMQZ,You bet! I'm here to assist you in adding prod...
3,di like to add products to the cart could i ge...,add_product,CART,BCILMPQZ,Indeed! I'm here to assist you in adding produ...
4,"I need to add an item to the cart , where do I...",add_product,CART,BCILZ,I'll take care of it! I'm here to help you wit...


In [ ]:
# save_dir = r"D:\giorg\corsi\ESGI\5eme_annee\NLP\projet_nlp\models\tnbert_cls\checkpoint-67350"#"../models/checkpoint-67350"

# tokenizer = AutoTokenizer.from_pretrained(save_dir)
# model = AutoModelForSequenceClassification.from_pretrained(save_dir)

# print(model.config.id2label)  # should show {'0': 'CART', '1': 'ORDER', '2': 'PRODUCT'} or similar
# print(model.config.label2id)  # should show {'0': 'CART', '1': 'ORDER', '2': 'PRODUCT'} or similar

# clf = pipeline(
#     "text-classification",
#     model=model,
#     tokenizer=tokenizer,
#     return_all_scores=False
# )

# text_pro = df[df['category'] == 'PRODUCT']['instruction'].sample(n=1).iloc[0]
# pred_pro = clf(text_pro)
# print(text_pro, "->", pred_pro)

Device set to use cuda:0


{0: 'CART', 1: 'PRODUCT', 2: 'ORDER', 3: 'ACCOUNT', 4: 'CONTACT', 5: 'DELIVERY', 6: 'PAYMENT', 7: 'RETURNS', 8: 'USER', 9: 'SALES', 10: 'STORE', 11: 'FEEDBACK', 12: 'APP_WEBSITE'}
{'ACCOUNT': 3, 'APP_WEBSITE': 12, 'CART': 0, 'CONTACT': 4, 'DELIVERY': 5, 'FEEDBACK': 11, 'ORDER': 2, 'PAYMENT': 6, 'PRODUCT': 1, 'RETURNS': 7, 'SALES': 9, 'STORE': 10, 'USER': 8}
I need to check the availability of a product in store, help me -> [{'label': 'USER', 'score': 0.999927282333374}]


d:\giorg\corsi\ESGI\5eme_annee\NLP\projet_nlp\venv\Lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [59]:
model_id = "gpasiniesgi/esgi_nlp_project_1"  # example HF repo with safetensors weights

clf = pipeline(
    task="text-classification",
    model=model_id,
    tokenizer=model_id,
    # use_safetensors is True by default for modern models, but you can be explicit:
    model_kwargs={"torch_dtype": "auto"},
)

text_pro = df[df['category'] == 'PRODUCT']['instruction'].sample(n=1).iloc[0]
pred_pro = clf(text_pro)

text_car = df[df['category'] == 'CART']['instruction'].sample(n=1).iloc[0]
pred_car = clf(text_car)

text_ord = df[df['category'] == 'ORDER']['instruction'].sample(n=1).iloc[0]
pred_ord = clf(text_ord)

print("product : " + text_pro)
print(f"product : {pred_pro}")

print(f'car : {text_car}')
print(f'car : {pred_car}')

print(f"ord : {text_ord}")
print(f"ord : {pred_ord}")


d:\giorg\corsi\ESGI\5eme_annee\NLP\projet_nlp\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\giorg\.cache\huggingface\hub\models--gpasiniesgi--esgi_nlp_project_1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fallin

product : I have to exchange a product I bought, can you help me ?
product : [{'label': 'PRODUCT', 'score': 0.999591052532196}]
car : could you help me to add a fucking item to the basket?
car : [{'label': 'CART', 'score': 0.9992533326148987}]
ord : i ordered a fucking product and i did not get a bill can i download it
ord : [{'label': 'ORDER', 'score': 0.9990365505218506}]


In [60]:

text_pro = df[df['category'] == 'PRODUCT']['instruction'].sample(n=1).iloc[0]
pred_pro = clf(text_pro)

text_car = df[df['category'] == 'CART']['instruction'].sample(n=1).iloc[0]
pred_car = clf(text_car)

text_ord = df[df['category'] == 'ORDER']['instruction'].sample(n=1).iloc[0]
pred_ord = clf(text_ord)

print("product : " + text_pro)
print(f"product : {pred_pro}")

print(f'car : {text_car}')
print(f'car : {pred_car}')

print(f"ord : {text_ord}")
print(f"ord : {pred_ord}")


# ['CART',
#  'PRODUCT',
#  'ORDER',
#  'ACCOUNT',
#  'CONTACT',
#  'DELIVERY',
#  'PAYMENT',
#  'RETURNS',
#  'USER',
#  'SALES',
#  'STORE',
#  'FEEDBACK',
#  'APP_WEBSITE']

product : I would like to check the fucking availability of an item, how could I do it?
product : [{'label': 'PRODUCT', 'score': 0.999645471572876}]
car : need to add a product to the basket where could i do it
car : [{'label': 'CART', 'score': 0.999257504940033}]
ord : i have to download aa fucking invoice can i get some help
ord : [{'label': 'ORDER', 'score': 0.9989069700241089}]
